# Selección de kernel y bandwidths ST-KDE mediante LOO-CV / ST-KDE Kernel and Bandwidth Selection Using LOO-CV
**[EN]**
This notebook documents the automatic selection of:

- **Kernel**: `gaussian`, `epanechnikov`, or `quartic`
- **Spatial bandwidth** $h_s$ (km)
- **Temporal bandwidth** $h_t$ (hours), evaluated on an annual scale

using **leave-one-out cross-validation (LOO-CV)** on `data/stkde_incidents.parquet`, which contains 41,935 incidents related to gender-based crimes occurring in public spaces.

Applying the pairwise procedure directly to all 41,935 records requires matrices with approximately 1.76 billion elements, as the algorithm has $O(n^2)$ computational complexity. To address this limitation, stratified samples of 400, 800, and 1,500 incidents are compared to determine the minimum sample size that yields stable kernel and bandwidth selection.

The candidate temporal bandwidth values correspond to **1, 1.5, and 2 years**. Since the dataset covers incidents from 2020 to 2025, observations become progressively less influential as their temporal distance from the evaluation time increases. Consequently, evaluating bandwidths beyond two years is unlikely to provide additional useful information.

**[ES]**

Este cuaderno documenta la selección automática de:

- **Kernel**: `gaussian`, `epanechnikov` o `quartic`
- **Bandwidth espacial** $h_s$ (km)
- **Bandwidth temporal** $h_t$ (horas), evaluado en escala anual

mediante **leave-one-out cross-validation (LOO-CV)** sobre `data/stkde_incidents.parquet`, que contiene  41 935 incidentes relacionados con delitos de género que suceden en el espacio público.

Usar el procedemiento pairwise directamente sobre los 41 935 registros exige matrices de aproximadamente 1.76 mil millones de elementos ya que tiene una complejidad algoritmica O(n^2). Por ello se comparan muestras estratificadas de 400, 800 y 1500 incidentes para encontrar el tamaño mínimo que estabiliza la selección.

Los valores temporales para el ancho de banda serán de  **1, 1.5 y 2 años**, ya que al tener delitos del periodo 2020-2025, cualquier fecha a partir del tiempo actual representara cada vez un valor con menor peso. 


## 0. Dependencias, rutas y rejilla de búsqueda / Dependencies, routes and search grid

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "app"))

# Se importa funciones y constantes el archivo stkde_model para esta evaluación / Import functions and constants from the stkde_model file for this evaluation.
 
from stkde_model import (
    COL_DATE,
    COL_HOUR,
    COL_LAT,
    COL_LON,
    KERNELS,
    TEMPORAL_BW_CANDIDATES_H,
    _build_pairwise_distances,
    _build_pairwise_temporal,
    _incident_datetimes,
    _loo_log_likelihood,
)

DATA_PATH = PROJECT_ROOT / "data" / "stkde_incidents.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data"

# Tamaños de muestra a comparar de manera estratificada / Sample sizes to be compared in a stratified manner
SAMPLE_SIZES = [400, 800, 1500]


# 1 año, 1.5 años y 2 años. Se importa del mismo código usado en producción. / 1 year, 1.5 years and 2 years. Imported from the same code used in production.
H_T_CANDIDATES = list(TEMPORAL_BW_CANDIDATES_H)  # 8760, 13140, 17520 h

# Rejilla espacial alrededor de la resolución de 500 m del mapa. / Spatial grid around the 500 m resolution of the map.
H_S_CANDIDATES = [0.5, 1.0, 1.5, 2.0, 3.0]

# Bloque de control: reproduce el h_s inicial antes de la búsqueda conjunta. /  Control block: plays the initial h_s before the joint search.
H_S_FIXED = 1.5

RANDOM_STATE = 42

print(f"Dataset: {DATA_PATH}")
print(f"Kernels: {list(KERNELS.keys())}")
print(f"h_t (h): {H_T_CANDIDATES}")
print(f"h_s (km): {H_S_CANDIDATES}")
print(f"Muestras: {SAMPLE_SIZES}")


## 1. Carga del dataset ST-KDE / Loading the ST-KDE Dataset 

**[EN]**

The dataset generated in `transformacion_datos_stkde.ipynb` is used, with the following columns:
`COORD. X`, `COORD. Y`, `FECHA DE LOS HECHOS`, `HORA DE LOS HECHOS`.

**[ES]**

Se usa el parquet generado en `transformacion_datos_stkde.ipynb`, con columnas:
`COORD. X`, `COORD. Y`, `FECHA DE LOS HECHOS`, `HORA DE LOS HECHOS`.


In [ ]:
df = pd.read_parquet(DATA_PATH)
df[COL_DATE] = df[COL_DATE].astype(str)
df[COL_HOUR] = df[COL_HOUR].astype(int)

print(f"Registros: {len(df):,}")
print(df.dtypes.to_string())
print(f"\nRango fechas: {df[COL_DATE].min()} — {df[COL_DATE].max()}")
df.head(3)


## 2. Se seleccionan tres muestras estratificadas (400, 800, 1500) / Three stratified samples (400, 800, 1500) are selected 

**[EN]**

Stratified sampling by latitude/longitude categories with sample sizes of 400, 800, 1500 and `random_state=42`, to fully cover Mexico City without bias towards a single hotspot and to allow for controlled comparison of sample sizes.

**[ES]**

Muestreo estratificado por categorias de latitud/longitud con tamaño 
$n 400, 800, 1500  y `random_state=42`, para cubrir de forma completa a la CDMX sin sesgar
hacia un solo hotspot y poder comparar tamaños de forma controlada.


In [ ]:
def stratified_sample(df: pd.DataFrame, n: int, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    """Muestra estratificada por bins lat/lon con tamaño exacto n / Sampling stratified by lat/lon bins with exact size n.""" 
    if len(df) <= n:
        return df.copy().reset_index(drop=True)

    work = df.copy()
    work["_lat_bin"] = pd.cut(work[COL_LAT], bins=8, labels=False)
    work["_lon_bin"] = pd.cut(work[COL_LON], bins=8, labels=False)
    work["_stratum"] = work["_lat_bin"].astype(str) + "_" + work["_lon_bin"].astype(str)

    counts = work["_stratum"].value_counts()
    quotas = (counts / counts.sum() * n).round().astype(int).clip(lower=1)
    while quotas.sum() > n:
        quotas.loc[quotas.idxmax()] -= 1
    while quotas.sum() < n:
        quotas.loc[quotas.idxmax()] += 1

    parts = []
    rng = random_state
    for stratum, quota in quotas.items():
        group = work[work["_stratum"] == stratum]
        take = min(len(group), int(quota))
        parts.append(group.sample(n=take, random_state=rng))
        rng += 1

    sampled = pd.concat(parts)
    if len(sampled) < n:
        remaining = work.drop(index=sampled.index)
        need = n - len(sampled)
        sampled = pd.concat([sampled, remaining.sample(n=need, random_state=random_state)])
    elif len(sampled) > n:
        sampled = sampled.sample(n=n, random_state=random_state)

    return (
        sampled.drop(columns=["_lat_bin", "_lon_bin", "_stratum"])
        .reset_index(drop=True)
    )


samples: dict[int, pd.DataFrame] = {}
for n in SAMPLE_SIZES:
    samples[n] = stratified_sample(df, n=n)
    print(
        f"n_objetivo={n:4d} → n_obtenido={len(samples[n]):4d} | "
        f"lat [{samples[n][COL_LAT].min():.3f}, {samples[n][COL_LAT].max():.3f}] | "
        f"lon [{samples[n][COL_LON].min():.3f}, {samples[n][COL_LON].max():.3f}]"
    )

assert all(len(samples[n]) == n for n in SAMPLE_SIZES), "Las muestras deben tener tamaño exacto" 


## 3. Matrices por pares: distancia (km) y t (horas) / Pairwise Matrices: Distance (km) and t (hours)

**[EN]**

For each sample, the following are constructed:

- `dist_km[i, j]`: Haversian distance between incidents $i$ and $j$
- `temp_hours[i, j]`: Absolute distance $|t_i - t_j|$ in hours

These matrices feed the Leave-One-Out (LOO): for point $i$, the distribution of all points $j$ is considered, provided that $i$ is different from $j$.

**[ES]**

Para cada muestra se construyen:

- `dist_km[i, j]`: distancia Haversiana entre incidentes $i$ y $j$
- `temp_hours[i, j]`: distancia absoluta $|t_i - t_j|$ en horas

Estas matrices alimentan el LOO(Leave-One-Out): para el punto $i$ se toman en cuenta la distribución de todos los puntos $j$ siempre
y cuando $i$ sea diferente de $j$.


In [ ]:
pairwise: dict[int, dict[str, np.ndarray]] = {}

for n, sample in samples.items():
    t0 = time.perf_counter()
    dist_km = _build_pairwise_distances(sample[COL_LAT].values, sample[COL_LON].values)
    temp_hours = _build_pairwise_temporal(_incident_datetimes(sample))
    pairwise[n] = {"dist_km": dist_km, "temp_hours": temp_hours}
    elapsed = time.perf_counter() - t0
    mem_mb = (dist_km.nbytes + temp_hours.nbytes) / (1024**2)
    print(
        f"n={n:4d} | matrices {dist_km.shape} | "
        f"memoria ≈ {mem_mb:.1f} MB | tiempo {elapsed:.2f} s"
    )
    print(
        f"         dist_km ∈ [{dist_km[dist_km > 0].min():.3f}, {dist_km.max():.3f}] km | "
        f"Δt ∈ [0, {temp_hours.max():.0f}] h"
    )


## 4. Búsqueda LOO-CV — kernel × $h_t$ (con $h_s = 1.5$ km fijo) / LOO-CV Search — kernel × $h_t$ (with $h_s = 1.5 km fixed) 

**[EN]**

This control block maintains $h_s = 1.5 km and evaluates the new annual timescale. 

```
for each sample n ∈ {400, 800, 1500}:

for each kernel ∈ {gaussian, epanechnikov, quartic}:

for each h_t ∈ {8760, 13140, 17520} hours:
calculate LOO log-likelihood (h_s = 1.5 km)
```

Equivalencies: 8760 h = 1 year; 13140 h = 1.5 years; 17520 h = 2 years.

The metric is the average of $\log \hat f_{-i}(x_i)$; **higher (less negative) is better**.

**[ES]**

Este bloque de control mantiene $h_s=1.5$ km y evalúa la nueva escala temporal anual.

```
para cada muestra n ∈ {400, 800, 1500}:
  para cada kernel ∈ {gaussian, epanechnikov, quartic}:
    para cada h_t ∈ {8760, 13140, 17520} horas:
      calcular LOO log-likelihood  (h_s = 1.5 km)
```

Equivalencias: 8760 h = 1 año; 13140 h = 1.5 años; 17520 h = 2 años.
La métrica es el promedio de $\log \hat f_{-i}(x_i)$; **mayor (menos negativo) es mejor**.


In [ ]:
def run_loo_grid(
    dist_km: np.ndarray,
    temp_hours: np.ndarray,
    h_s_candidates: list[float],
    h_t_candidates: list[float],
) -> pd.DataFrame:
    rows = []
    for kernel_name, (kernel_fn, _) in KERNELS.items():
        for h_s in h_s_candidates:
            for h_t in h_t_candidates:
                ll = _loo_log_likelihood(dist_km, temp_hours, kernel_fn, h_s, h_t)
                rows.append(
                    {
                        "kernel": kernel_name,
                        "h_s_km": h_s,
                        "h_t_hours": h_t,
                        "loo_log_likelihood": ll,
                    }
                )
    return pd.DataFrame(rows)


results_fixed_hs: dict[int, pd.DataFrame] = {}
winners_fixed_hs = []

for n in SAMPLE_SIZES:
    t0 = time.perf_counter()
    grid = run_loo_grid(
        pairwise[n]["dist_km"],
        pairwise[n]["temp_hours"],
        h_s_candidates=[H_S_FIXED],
        h_t_candidates=H_T_CANDIDATES,
    )
    grid.insert(0, "n_sample", n)
    results_fixed_hs[n] = grid
    best = grid.loc[grid["loo_log_likelihood"].idxmax()]
    winners_fixed_hs.append(best)
    print(
        f"n={n:4d} | mejor: kernel={best['kernel']:12s} "
        f"h_s={best['h_s_km']} km  h_t={best['h_t_hours']:.0f} h  "
        f"LL={best['loo_log_likelihood']:.4f}  ({time.perf_counter() - t0:.1f} s)"
    )

summary_fixed = pd.DataFrame(winners_fixed_hs).reset_index(drop=True)
print("\nGanadores (h_s fijo = 1.5 km):")
summary_fixed


In [ ]:
best_by_kernel = []
for n, grid in results_fixed_hs.items():
    for kernel, g in grid.groupby("kernel"):
        row = g.loc[g["loo_log_likelihood"].idxmax()]
        best_by_kernel.append(
            {
                "n_sample": n,
                "kernel": kernel,
                "best_h_t": row["h_t_hours"],
                "loo_ll": row["loo_log_likelihood"],
            }
        )

best_by_kernel_df = pd.DataFrame(best_by_kernel)
print("Mejor h_t por kernel y tamaño de muestra (h_s=1.5 km):")
best_by_kernel_df.pivot(index="kernel", columns="n_sample", values=["best_h_t", "loo_ll"])


## 5. Búsqueda conjunta — kernel × $h_s$ × $h_t$ / Joint Search — kernel × $h_s$ × $h_t$

**[EN]**

In this block, both bandwidths are selected simultaneously:

```
for each sample n ∈ {400, 800, 1500}:
for each kernel ∈ {gaussian, epanechnikov, quartic}:
for each h_s ∈ {0.5, 1.0, 1.5, 2.0, 3.0} km:
for each h_t ∈ {8760, 13140, 17520} hours:
calculate LOO log-likelihood
```

The winning configuration is the trio $(kernel,h_s,h_t)$ with the highest LOO log-likelihood. In the recalibration of the project, the winner was **gaussian, $h_s=3$ km and $h_t=8760$ h (1 year)**

**[ES]**

En este bloque ambos bandwidths se seleccionan simultáneamente:

```
para cada muestra n ∈ {400, 800, 1500}:
  para cada kernel ∈ {gaussian, epanechnikov, quartic}:
    para cada h_s ∈ {0.5, 1.0, 1.5, 2.0, 3.0} km:
      para cada h_t ∈ {8760, 13140, 17520} horas:
        calcular LOO log-likelihood
```

La configuración ganadora es el trío $(kernel,h_s,h_t)$ con mayor LOO log-likelihood. En la recalibración del proyecto, el ganador fue **gaussian, $h_s=3$ km y $h_t=8760$ h (1 año)**.


In [ ]:
results_joint: dict[int, pd.DataFrame] = {}
winners_joint = []

for n in SAMPLE_SIZES:
    t0 = time.perf_counter()
    grid = run_loo_grid(
        pairwise[n]["dist_km"],
        pairwise[n]["temp_hours"],
        h_s_candidates=H_S_CANDIDATES,
        h_t_candidates=H_T_CANDIDATES,
    )
    grid.insert(0, "n_sample", n)
    results_joint[n] = grid
    best = grid.loc[grid["loo_log_likelihood"].idxmax()]
    winners_joint.append(best)
    print(
        f"n={n:4d} | mejor: kernel={best['kernel']:12s} "
        f"h_s={best['h_s_km']} km  h_t={best['h_t_hours']:.0f} h  "
        f"LL={best['loo_log_likelihood']:.4f}  ({time.perf_counter() - t0:.1f} s)"
    )

summary_joint = pd.DataFrame(winners_joint).reset_index(drop=True)
print("\nGanadores (búsqueda conjunta h_s × h_t):")
summary_joint


## 6. Comparación entre muestras y número mínimo de incidentes / Comparison between samples and minimum number of incidents

**[EN]**

**Stability criterion:** The minimum sample size $n^*$ is the smallest $n$ of \{400, 800, 1500\}
whose winner $(\text{kernel}, h_s, h_t)$ matches that of the reference sample
($n=1500$, since it is the largest and therefore the most reliable within the experiment).

If 400 crimes already matches 1500 crimes, then **a sample of 400 is sufficient to use** (and is the minimum
proven to stabilize). If it only stabilizes from 800 onwards, the minimum is 800, etc.

**[ES]**

**Criterio de estabilidad:** el tamaño mínimo de la muestra $n^*$ es el menor $n$ de \{400, 800, 1500\}
cuyo ganador $(\text{kernel}, h_s, h_t)$ coincide con el de la muestra de referencia
($n=1500$, ya que es la más grande y por tanto la más fiable dentro del experimento).

Si 400 delitos ya coincide con 1500 delitos, entonces **una muestra 400 es suficiente para usar** (y es el mínimo
probado que estabiliza). Si solo estabiliza a partir de 800, el mínimo es 800, etc.


In [ ]:
def config_key(row: pd.Series) -> tuple:
    return (row["kernel"], float(row["h_s_km"]), float(row["h_t_hours"]))


def find_minimum_stable_n(summary: pd.DataFrame, label: str) -> int:
    ref_n = max(SAMPLE_SIZES)
    ref = summary.loc[summary["n_sample"] == ref_n].iloc[0]
    ref_key = config_key(ref)
    print(f"\n=== {label} ===")
    print(f"Referencia n={ref_n}: {ref_key}  LL={ref['loo_log_likelihood']:.4f}")

    min_stable = None
    for n in SAMPLE_SIZES:
        row = summary.loc[summary["n_sample"] == n].iloc[0]
        key = config_key(row)
        match = key == ref_key
        mark = "✓ coincide con n=1500" if match else "✗ difiere"
        print(f"  n={n:4d}: {key}  LL={row['loo_log_likelihood']:.4f}  {mark}")
        if match and min_stable is None:
            min_stable = n

    assert min_stable is not None, "Ninguna muestra coincidió con la referencia"
    print(f"→ Número mínimo estable: n* = {min_stable}")
    return min_stable


n_star_fixed = find_minimum_stable_n(summary_fixed, "Bloque A: h_s = 1.5 km fijo")
n_star_joint = find_minimum_stable_n(summary_joint, "Bloque B: búsqueda conjunta h_s × h_t")

print("\n=== Margen del ganador vs 2º lugar (n=1500, conjunto) ===")
top1500 = results_joint[1500].sort_values("loo_log_likelihood", ascending=False)
print(top1500.head(5).to_string(index=False))
gap = top1500.iloc[0]["loo_log_likelihood"] - top1500.iloc[1]["loo_log_likelihood"]
print(f"\nGap 1º–2º: {gap:.4f} (mayor gap ⇒ ranking más claro)")


In [ ]:
all_fixed = pd.concat(results_fixed_hs.values(), ignore_index=True)
all_joint = pd.concat(results_joint.values(), ignore_index=True)

path_fixed = OUTPUT_DIR / "loo_cv_results_hs_fixed.csv"
path_joint = OUTPUT_DIR / "loo_cv_results_joint.csv"
path_summary = OUTPUT_DIR / "loo_cv_winners_summary.csv"

all_fixed.to_csv(path_fixed, index=False)
all_joint.to_csv(path_joint, index=False)

summary_export = pd.concat(
    [
        summary_fixed.assign(search="hs_fixed_1.5"),
        summary_joint.assign(search="joint_hs_ht"),
    ],
    ignore_index=True,
)
summary_export.to_csv(path_summary, index=False)

print(f"Guardado: {path_fixed.name}")
print(f"Guardado: {path_joint.name}")
print(f"Guardado: {path_summary.name}")


## 7. Justificación final — parámetros para el proyecto / Final Justification — Parameters for the Project

**[EN]**

This section consolidates the decision to select the parameters based on the results calculated above (these can be replicated by re-running the workbook).

**[ES]**

Esta sección consolida la decisión de selección de los parametros a partir de los resultados 
calculados arriba(se pueden replicar re-ejecutando el cuaderno).


In [ ]:
ref = summary_joint.loc[summary_joint["n_sample"] == 1500].iloc[0]
chosen_kernel = ref["kernel"]
chosen_hs = float(ref["h_s_km"])
chosen_ht = float(ref["h_t_hours"])
chosen_ll = float(ref["loo_log_likelihood"])

n_star = max(n_star_fixed, n_star_joint)

print("=" * 70)
print("DECISIÓN PARA EL PROYECTO ST-KDE")
print("=" * 70)
print(
    f"""
1) Número mínimo de incidentes para LOO-CV
   - Bloque A (h_s fijo 1.5 km): n* = {n_star_fixed}
   - Bloque B (h_s × h_t conjunto): n* = {n_star_joint}
   - Recomendación: usar al menos n = {n_star} en la selección de parámetros.
     Motivo: es el menor tamaño (entre 400, 800 y 1500) que reproduce el mismo
     ganador que la muestra de referencia n=1500 bajo el criterio de estabilidad.

2) Parámetros óptimos según LOO log-likelihood (referencia n=1500, búsqueda conjunta)
   - Kernel:              {chosen_kernel}
   - h_s (espacial):      {chosen_hs} km
   - h_t (temporal):      {chosen_ht:.0f} h  ({chosen_ht/8760:.2f} años)
   - LOO log-likelihood:  {chosen_ll:.4f}

3) Lectura metodológica
   - Se elige el trío (kernel, h_s, h_t) con mayor LOO-LL en n=1500.
   - La rejilla temporal de 1–2 años evita que consultas posteriores a 2025
     pierdan todo el peso temporal, sin fijar arbitrariamente un valor de 3 años.
   - Se valida que muestras más pequeñas no cambien esa decisión;
     el n* indicado es el mínimo empírico que estabiliza el ranking.
   - Los ~41 935 registros se usan en la estimación final; el submuestreo
     solamente selecciona los hiperparámetros.
"""
)

epa = results_joint[1500]
epa = epa[epa["kernel"] == "epanechnikov"].sort_values("loo_log_likelihood", ascending=False)
gau = results_joint[1500]
gau = gau[gau["kernel"] == "gaussian"].sort_values("loo_log_likelihood", ascending=False)
print("Mejor configuración gaussiana vs Epanechnikov (n=1500):")
print(
    f"  gaussian     → h_s={gau.iloc[0]['h_s_km']}  h_t={gau.iloc[0]['h_t_hours']:.0f}  "
    f"LL={gau.iloc[0]['loo_log_likelihood']:.4f}"
)
print(
    f"  epanechnikov → h_s={epa.iloc[0]['h_s_km']}  h_t={epa.iloc[0]['h_t_hours']:.0f}  "
    f"LL={epa.iloc[0]['loo_log_likelihood']:.4f}"
)
print(
    f"  Δ LL (gauss - epa) = {gau.iloc[0]['loo_log_likelihood'] - epa.iloc[0]['loo_log_likelihood']:.4f}"
)


## Resumen y decisión final / Summary and Final Decision

**[EN]**

| Step | Description                                                                     |
| ---- | ------------------------------------------------------------------------------- |
| 1–2  | Load `stkde_incidents.parquet` and generate stratified samples (400, 800, 1500) |
| 3    | Pairwise Haversine (km) and Δt (hours) distance matrices                        |
| 4    | LOO-CV: kernel × annual $h_t$ with $h_s = 1.5$ km                               |
| 5    | Joint LOO-CV: kernel × $h_s$ × $h_t$                                            |
| 6    | Stability criterion → minimum $n^*$                                             |
| 7    | Final parameter selection based on the maximum LOO log-likelihood               |

The current configuration, serialized in `app/stkde_config.json`, is:

* Kernel: **gaussian**
* $h_s$: **3.0 km**
* $h_t$: **8760 h (1 year)**
* LOO log-likelihood (current fit): **−14.3437**

The **1-year** temporal bandwidth outperformed the **1.5-year** and **2-year** alternatives; therefore, a **3-year** bandwidth was not selected manually. The **Low/Medium/High** risk thresholds are calibrated afterward using the **6,090 grid cells** and are documented in `clasificacion_riesgo_grilla.ipynb`.

Artifacts exported to `data/`:
`loo_cv_results_hs_fixed.csv`, `loo_cv_results_joint.csv`, `loo_cv_winners_summary.csv`.

**[ES]**

| Paso | Contenido |
|---|---|
| 1–2 | Carga de `stkde_incidents.parquet` y muestras estratificadas (400, 800, 1500) |
| 3 | Matrices pairwise Haversine (km) y Δt (horas) |
| 4 | LOO-CV: kernel × $h_t$ anual con $h_s=1.5$ km |
| 5 | LOO-CV conjunto: kernel × $h_s$ × $h_t$ |
| 6 | Criterio de estabilidad → $n^*$ mínimo |
| 7 | Parámetros finales por máximo LOO-LL |

La configuración vigente serializada en `app/stkde_config.json` es:

- kernel: **gaussian**
- $h_s$: **3.0 km**
- $h_t$: **8760 h (1 año)**
- LOO log-likelihood (ajuste vigente): **−14.3437**

La selección de 1 año superó a 1.5 y 2 años; por eso no se fija 3 años manualmente. Los umbrales Bajo/Medio/Alto se calibran después sobre las 6 090 celdas y se documentan en `clasificacion_riesgo_grilla.ipynb`.

Artefactos exportados en `data/`:
`loo_cv_results_hs_fixed.csv`, `loo_cv_results_joint.csv`, `loo_cv_winners_summary.csv`.
